# Project: Mini-Code Autocompleter & Syntax Validator (Causal Decoder-Only Transformer)

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

In [2]:
np.random.seed(42)
tf.random.set_seed(42)

# Step 1: Synthetic Dataset & Tokenization
names = ['calc', 'process', 'run', 'eval', 'step']
args = ['x', 'y', 'z', 'a', 'b']

lines = []
for _ in range(2500):
    name = np.random.choice(names)
    arg = np.random.choice(args)
    const = np.random.randint(1,10)
    lines.append(f"fn {name} ({arg}): return {arg} +{const};")

all_texts = " ".join(lines)
print("Corpus Sample: \n", all_texts[:120])

# Vocabulary and mapping
vocab = sorted(list(set(all_texts)))
vocab_size = len(vocab)
char_to_idx = {ch: i for i, ch in enumerate(vocab)}
idx_to_char = {i: ch for i, ch in enumerate(vocab)}

print(f"\nVocabulary Size: {vocab_size}")
print("Vocabulary: ", vocab)

# Integer encoding
data_indices = np.array([char_to_idx[c] for c in all_texts], dtype=np.int32)

# Create shifted sequences
seq_len = 32
step = 3
inputs_list, targets_list = [],[]

for i in range(0, len(data_indices) - seq_len - 1, step):
    inputs_list.append(data_indices[i: i + seq_len])
    targets_list.append(data_indices[i + 1 : i + seq_len + 1])

x_train = np.array(inputs_list, dtype=np.int32)
y_train = np.array(targets_list, dtype=np.int32)

print(f" Shape: X_Train -> {x_train.shape}, Y_Train -> {y_train.shape}")

Corpus Sample: 
 fn eval (b): return b +8; fn step (b): return b +7; fn process (z): return z +7; fn run (z): return z +8; fn step (a): r

Vocabulary Size: 32
Vocabulary:  [' ', '(', ')', '+', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', 'a', 'b', 'c', 'e', 'f', 'l', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'x', 'y', 'z']
 Shape: X_Train -> (22001, 32), Y_Train -> (22001, 32)


In [3]:
class GPTEmbedding(tf.keras.layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_emb = tf.keras.layers.Embedding(vocab_size, embed_dim)
        self.pos_emb = tf.keras.layers.Embedding(maxlen, embed_dim)
        self.maxlen = maxlen

    def call(self, x):
        length = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=length, delta=1)
        return self.token_emb(x) + self.pos_emb(positions)

In [4]:
class CausalTransformerBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, d_ff, dropout_rate=0.01, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model 
        self.num_heads = num_heads
        self.d_ff = d_ff

        self.mha = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model // num_heads
        )

        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(d_ff, activation='relu'),
            tf.keras.layers.Dense(d_model)
        ])

        self.ln1 = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.ln2 = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.dropout1 = tf.keras.layers.Dropout(dropout_rate)
        self.dropout2 = tf.keras.layers.Dropout(dropout_rate)

    def call(self, inputs, training=False):
        norm1 = self.ln1(inputs)
        attn_out = self.mha(
            query=norm1,
            key=norm1,
            value=norm1,
            use_causal_mask=True
        )
        x = inputs + self.dropout1(attn_out, training=training)

        norm2 = self.ln2(x)
        ffn_out = self.ffn(norm2)
        x = x + self.dropout2(ffn_out, training=training)

        return x

In [5]:
embed_dim = 64
num_heads = 4
d_ff = 128

inputs = tf.keras.layers.Input(shape=(seq_len,), dtype = tf.int32)
x = GPTEmbedding(maxlen = seq_len, vocab_size=vocab_size, embed_dim=embed_dim)(inputs)

x = CausalTransformerBlock(d_model=embed_dim, num_heads=num_heads, d_ff=d_ff)(x)
x = CausalTransformerBlock(d_model=embed_dim, num_heads=num_heads, d_ff=d_ff)(x)

x = tf.keras.layers.LayerNormalization(epsilon=1e-5)(x)
outputs = tf.keras.layers.Dense(vocab_size, activation="softmax")(x)

gpt_model = tf.keras.Model(inputs=inputs, outputs=outputs)

In [6]:
gpt_model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.01),
    loss = "sparse_categorical_crossentropy",
    metrics = ['accuracy']
)

In [7]:
gpt_model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ gpt_embedding (GPTEmbedding)         │ (None, 32, 64)              │           4,096 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ causal_transformer_block             │ (None, 32, 64)              │          33,472 │
│ (CausalTransformerBlock)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ causal_transformer_block_1           │ (None, 32, 64)              │          33,472 │
│ (CausalTransformerBlock)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ layer_normalization_4                │ (None, 32, 64)              │             128 │
│ (LayerNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 32, 32)              │           2,080 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 73,248 (286.12 KB)

 Trainable params: 73,248 (286.12 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
history = gpt_model.fit(
    x_train, y_train,
    batch_size=64,
    epochs=15,
    verbose=1
)

Epoch 1/15
344/344 ━━━━━━━━━━━━━━━━━━━━ 39s 62ms/step - accuracy: 0.8380 - loss: 0.4409
Epoch 2/15
344/344 ━━━━━━━━━━━━━━━━━━━━ 22s 63ms/step - accuracy: 0.8833 - loss: 0.2701
Epoch 3/15
344/344 ━━━━━━━━━━━━━━━━━━━━ 21s 62ms/step - accuracy: 0.8865 - loss: 0.2564
Epoch 4/15
344/344 ━━━━━━━━━━━━━━━━━━━━ 21s 62ms/step - accuracy: 0.8869 - loss: 0.2547
Epoch 5/15
344/344 ━━━━━━━━━━━━━━━━━━━━ 21s 60ms/step - accuracy: 0.8839 - loss: 0.2652
Epoch 6/15
344/344 ━━━━━━━━━━━━━━━━━━━━ 21s 60ms/step - accuracy: 0.8849 - loss: 0.2616
Epoch 7/15
344/344 ━━━━━━━━━━━━━━━━━━━━ 21s 61ms/step - accuracy: 0.8865 - loss: 0.2543
Epoch 8/15
344/344 ━━━━━━━━━━━━━━━━━━━━ 41s 60ms/step - accuracy: 0.8865 - loss: 0.2539
Epoch 9/15
344/344 ━━━━━━━━━━━━━━━━━━━━ 21s 61ms/step - accuracy: 0.8866 - loss: 0.2536
Epoch 10/15
344/344 ━━━━━━━━━━━━━━━━━━━━ 21s 61ms/step - accuracy: 0.8845 - loss: 0.2624
Epoch 11/15
344/344 ━━━━━━━━━━━━━━━━━━━━ 21s 60ms/step - accuracy: 0.8787 - loss: 0.2828
Epoch 12/15
344/344 ━━━━━━━━━━

In [9]:
def complete_code(prompt, max_new_tokens=30, temperature=0.0):
    generated = [char_to_idx[c] for c in prompt]
    
    for _ in range(max_new_tokens):
        # 1. Context window (seq_len = 32)
        ctx = generated[-seq_len:]
        if len(ctx) < seq_len:
            ctx = [0] * (seq_len - len(ctx)) + ctx
            
        inp = np.array([ctx], dtype=np.int32)
        
        # 2. Forward pass
        preds = gpt_model.predict(inp, verbose=0)
        last_logits = preds[0, -1, :]
        
        # 3. Greedy selection
        if temperature == 0.0:
            next_idx = int(np.argmax(last_logits))
        else:
            # Scaled sampling
            logits = np.log(last_logits + 1e-9) / temperature
            probs = np.exp(logits) / np.sum(np.exp(logits))
            next_idx = int(np.random.choice(len(probs), p=probs))
            
        generated.append(next_idx)
        
        # Stop at semicolon
        if idx_to_char[next_idx] == ';':
            break
            
    return "".join([idx_to_char[i] for i in generated])

In [10]:
test_prompts = [
    "fn calc(x): return",
    "fn process(a): return",
    "fn step(z):",
    "fn eval(b): return b +"
]

for p in test_prompts:
    res = complete_code(p, temperature=0.0)
    print(f"Prompt:     '{p}'")
    print(f"Completed:  '{res}'\n")

Prompt:     'fn calc(x): return'
Completed:  'fn calc(x): return x +6;'

Prompt:     'fn process(a): return'
Completed:  'fn process(a): return a +6;'

Prompt:     'fn step(z):'
Completed:  'fn step(z): return z +6;'

Prompt:     'fn eval(b): return b +'
Completed:  'fn eval(b): return b +6;'



In [11]:
import re

def validate_code_completion(completed_str):
    # Regex pattern: fn <name>(<arg>): return <arg> + <digit>;
    pattern = r"^fn\s+([a-zA-Z]+)\(([a-zA-Z]+)\):\s+return\s+([a-zA-Z]+)\s*\+\s*(\d+);$"
    match = re.match(pattern, completed_str.strip())
    
    if not match:
        return False, "Syntax error: Does not match standard function structure."
    
    func_name, declared_arg, returned_arg, const_val = match.groups()
    
    # Check variable scoping
    if declared_arg != returned_arg:
        return False, f"Scope error: Declared '{declared_arg}', but returned '{returned_arg}'."
        
    return True, f"Valid! Defined '{func_name}({declared_arg})' and correctly returned '{returned_arg} + {const_val}'."

# Test the completions
completions = [
    'fn calc(x): return x +6;',
    'fn process(a): return a +6;',
    'fn step(z): return z +6;',
    'fn eval(b): return b +6;'
]

for code in completions:
    is_valid, msg = validate_code_completion(code)
    status = "PASS" if is_valid else "FAIL"
    print(f"[{status}] {code} -> {msg}")

[PASS] fn calc(x): return x +6; -> Valid! Defined 'calc(x)' and correctly returned 'x + 6'.
[PASS] fn process(a): return a +6; -> Valid! Defined 'process(a)' and correctly returned 'a + 6'.
[PASS] fn step(z): return z +6; -> Valid! Defined 'step(z)' and correctly returned 'z + 6'.
[PASS] fn eval(b): return b +6; -> Valid! Defined 'eval(b)' and correctly returned 'b + 6'.
